# Train Sentiment Analyst in Google Colab

This notebook trains a model and creates evaluation reports from Sentiment140, Keras IMDB, or the sample dataset. Direct uploads from Colab need a `GITHUB_TOKEN` secret. If it is missing, the save cell will point you to the token-free GitHub Actions workflow.


In [2]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/content/SentimentalAnalyst')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/isravel-eng/SentimentalAnalyst.git', str(REPO_DIR)], check=True)
%cd /content/SentimentalAnalyst
%pip install -q -r requirements.txt

/content/SentimentalAnalyst


## Train and review

The default trains on a balanced sample of up to 100,000 Sentiment140 tweets. Set `DATASET = 'imdb'` for Keras movie reviews or `DATASET = 'sample'` for the included three-class demo. Sentiment140 and IMDB are binary datasets; neither has neutral training examples. Set `MAX_SAMPLES = 0` to use all Sentiment140 tweets.

In [ ]:
DATASET = 'sentiment140'
MAX_SAMPLES = 100000
if DATASET == 'imdb':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-imdb.txt'], check=True)

subprocess.run([
    sys.executable, 'main.py', 'train', '--dataset', DATASET,
    '--max-samples', str(MAX_SAMPLES),
    '--model', 'models/sentiment_model.pkl',
    '--report', 'reports/evaluation.txt',
    '--image-report', 'reports/evaluation_report.png',
], check=True)

from IPython.display import Image, display
display(Image('reports/evaluation_report.png'))

## Save the model and reports to GitHub

If `GITHUB_TOKEN` is configured in Colab Secrets, this cell uploads the model and reports to `main`. If not, it exits with instructions for the GitHub Actions workflow, which trains and saves the same artifacts without a Colab token.


In [ ]:
import base64
from pathlib import Path

import requests
from google.colab import userdata

OWNER, REPO, BRANCH = 'isravel-eng', 'SentimentalAnalyst', 'main'

try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None

if not TOKEN:
    print('No GITHUB_TOKEN Colab Secret was found. The Colab training files were not uploaded.')
    print('To save the artifacts without a Colab token, open this repository’s Actions tab,')
    print('select “Train sentiment model”, and choose “Run workflow” on main:')
    print(f'https://github.com/{OWNER}/{REPO}/actions/workflows/train-sentiment.yml')
else:
    headers = {
        'Authorization': f'Bearer {TOKEN}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }

    def upload_to_github(local_path, repo_path):
        local_path = Path(local_path)
        url = f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{repo_path}'
        current = requests.get(url, headers=headers, params={'ref': BRANCH}, timeout=30)
        payload = {
            'message': f'Update training artifact: {repo_path}',
            'content': base64.b64encode(local_path.read_bytes()).decode('ascii'),
            'branch': BRANCH,
        }
        if current.status_code == 200:
            payload['sha'] = current.json()['sha']
        elif current.status_code != 404:
            current.raise_for_status()
        response = requests.put(url, headers=headers, json=payload, timeout=120)
        response.raise_for_status()
        print(f'Saved {repo_path} to GitHub')

    for local_path in (
        'models/sentiment_model.pkl',
        'reports/evaluation.txt',
        'reports/evaluation_report.png',
    ):
        upload_to_github(local_path, local_path)


## Deploy the app

After the upload cell succeeds, create an app on Streamlit Community Cloud using this repository, the `main` branch, and `app.py`. The app will load the model file that Colab just saved.